# Round 5: Can one vector replace fine-tuning?

**The question:** fine-tuning moved Qwen2.5-0.5B from 6.5% → 33% on GSM8K by nudging millions of weights. But maybe most of that nudge collapses to a single *direction* in activation space. We test it:

1. Compute the **steering vector**: mean(fine-tuned activations − base activations) at layer 12, over ~50k tokens of math text. One 896-dim vector.
2. Inject it into the **base model** (no training, no weight changes) during generation: every token's layer-12 output gets `+ alpha * vector`.
3. Evaluate on the same 200 held-out problems at several strengths (alpha).
4. **Variant B (the novel one):** filter the vector through the round-4 SAE — keep only the top-K most-shifted features (the division detector, the 'Answer:' feature, etc.), discard the rest as noise. Sparse, interpretable steering.

**Scoreboard to beat:** base 6.5%, full fine-tune 33.0%. Where does steering land?

**Honest framing:** activation steering and task vectors exist in research. Comparing a raw fine-tuning-delta steer against an SAE-denoised steer, on a home-trained SAE, at this scale — that specific comparison is plausibly unexplored.

Setup: T4 GPU, Run all. ~1.5 hr fresh (includes retraining adapter + SAE if missing).

In [1]:
# Cell 1: Install
!pip uninstall -y -q torchao
!pip install -q -U transformers datasets peft accelerate

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.7 MB/s eta 0:00:00
GPU: Tesla T4


In [2]:
# Cell 2: Data + tokenizer
import random, re, os, gc
from datasets import load_dataset
from transformers import AutoTokenizer

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MAX_LEN = 640
LAYER = 12

torch.manual_seed(SEED)
gsm = load_dataset("openai/gsm8k", "main")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt(q):
    return f"Question: {q}\nAnswer:"

train_pool = list(gsm["train"])
random.Random(SEED).shuffle(train_pool)
train_pool = train_pool[:2000]

steer_pool = list(gsm["train"])[3000:3300]   # 300 TRAIN problems for building the vector
eval_pool = list(gsm["test"])[:200]           # the SAME 200 test problems as every round
print(f"Vector-building set: {len(steer_pool)} (train split) | eval: {len(eval_pool)} (test split)")
print("Note: the steering vector is built from TRAIN data only — no test leakage.")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vector-building set: 300 (train split) | eval: 200 (test split)
Note: the steering vector is built from TRAIN data only — no test leakage.


In [3]:
# Cell 3: Ensure the fine-tuned adapter exists (retrains if missing, ~20 min)
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

ADAPTER = "./adapter_r3_A_shuffled"

if not os.path.exists(ADAPTER):
    print("Adapter missing — retraining (~20 min)...")
    def tokenize_masked(ex):
        answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
        prompt = make_prompt(ex["question"])
        full = prompt + " " + answer + tokenizer.eos_token
        p_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
        enc = tokenizer(full, truncation=True, max_length=MAX_LEN)
        labels = list(enc["input_ids"])
        n = min(len(p_ids), len(labels))
        labels[:n] = [-100] * n
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

    order = list(train_pool)
    random.Random(SEED + 1).shuffle(order)
    data = [tokenize_masked(e) for e in order]

    class LD(Dataset):
        def __init__(s, it): s.it = it
        def __len__(s): return len(s.it)
        def __getitem__(s, i): return s.it[i]

    def coll(batch):
        m = max(len(x["input_ids"]) for x in batch)
        pid = tokenizer.pad_token_id
        ii, am, lb = [], [], []
        for x in batch:
            n = m - len(x["input_ids"])
            ii.append(list(x["input_ids"]) + [pid]*n)
            am.append(list(x["attention_mask"]) + [0]*n)
            lb.append(list(x["labels"]) + [-100]*n)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(lb)}

    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                     task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj"]))
    args = TrainingArguments(output_dir="./out_r5", num_train_epochs=1, per_device_train_batch_size=4,
                             gradient_accumulation_steps=2, learning_rate=1e-4, lr_scheduler_type="constant",
                             warmup_steps=10, logging_steps=100, save_strategy="no", fp16=True,
                             report_to="none", seed=SEED)
    Trainer(model=model, args=args, train_dataset=LD(data), data_collator=coll).train()
    model.save_pretrained(ADAPTER)
    del model; gc.collect(); torch.cuda.empty_cache()

print("✓ Adapter ready")

Adapter missing — retraining (~20 min)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Step,Training Loss
100,0.575378
200,0.553554


✓ Adapter ready


In [4]:
# Cell 4: Build the steering vector — mean activation difference at layer 12 (~10 min)
def get_layer_list(model):
    inner = model
    while not hasattr(inner, "layers"):
        for attr in ("base_model", "model"):
            if hasattr(inner, attr):
                inner = getattr(inner, attr); break
        else:
            raise RuntimeError("no layers found")
    return inner.layers

def capture_mean_and_acts(use_adapter, keep_acts=False):
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, ADAPTER) if use_adapter else base
    model.eval()
    captured = []
    def hook(m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        captured.append(h.detach().float().cpu())
    handle = get_layer_list(model)[LAYER].register_forward_hook(hook)

    total, count, acts_list = None, 0, []
    with torch.no_grad():
        for pi, ex in enumerate(steer_pool):
            answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
            text = make_prompt(ex["question"]) + " " + answer
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=200).to("cuda")
            captured.clear()
            model(**enc)
            a = captured[0][0]                    # [seq, 896]
            total = a.sum(0) if total is None else total + a.sum(0)
            count += a.shape[0]
            if keep_acts: acts_list.append(a)
            if (pi+1) % 100 == 0: print(f"  {pi+1}/{len(steer_pool)}")
    handle.remove()
    del model, base; gc.collect(); torch.cuda.empty_cache()
    mean = total / count
    return (mean, torch.cat(acts_list, 0) if keep_acts else None)

print("Base model pass...")
mean_base, acts_base = capture_mean_and_acts(False, keep_acts=True)
print("Fine-tuned model pass...")
mean_ft, acts_ft = capture_mean_and_acts(True, keep_acts=True)

steer_vec = (mean_ft - mean_base)
print(f"\nSteering vector built. Norm: {steer_vec.norm():.3f}")
print(f"(typical activation norm: {mean_base.norm():.3f} — the delta is a small nudge, as expected from LoRA)")

Base model pass...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  100/300
  200/300
  300/300
Fine-tuned model pass...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  100/300
  200/300
  300/300

Steering vector built. Norm: 0.562
(typical activation norm: 14.735 — the delta is a small nudge, as expected from LoRA)


In [5]:
# Cell 5: Steered evaluation machinery — inject vector at layer 12 during generation
def extract_gold(t): return t.split("####")[-1].strip().replace(",", "")
def extract_pred(g):
    g = g.split("Question:")[0]
    if "####" in g:
        nums = re.findall(r"-?\d+\.?\d*", g.split("####")[-1].replace(",", ""))
        if nums: return nums[0].rstrip(".")
    nums = re.findall(r"-?\d+\.?\d*", g.replace(",", ""))
    return nums[-1].rstrip(".") if nums else None
def norm(x):
    try: return float(x)
    except (TypeError, ValueError): return None

@torch.no_grad()
def evaluate_steered(vector, alpha, label, n_eval=200, batch_size=8, use_adapter=False):
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, ADAPTER) if use_adapter else base
    model.eval()

    handle = None
    if vector is not None and alpha != 0:
        v = (alpha * vector).to("cuda", dtype=torch.float16)
        def hook(m, i, o):
            if isinstance(o, tuple):
                return (o[0] + v,) + o[1:]
            return o + v
        handle = get_layer_list(model)[LAYER].register_forward_hook(hook)

    pool = eval_pool[:n_eval]
    tokenizer.padding_side = "left"
    correct = 0
    for i in range(0, len(pool), batch_size):
        batch = pool[i:i+batch_size]
        prompts = [make_prompt(ex["question"]) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to("cuda")
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j, ex in enumerate(batch):
            gen = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            if norm(extract_pred(gen)) is not None and norm(extract_pred(gen)) == norm(extract_gold(ex["answer"])):
                correct += 1
    if handle: handle.remove()
    del model, base; gc.collect(); torch.cuda.empty_cache()
    acc = correct / len(pool)
    print(f"{label:<40} {correct}/{len(pool)} = {acc:.1%}")
    return acc

print("Machinery ready.")

Machinery ready.


In [6]:
# Cell 6: EXPERIMENT A — raw steering vector, alpha sweep on 100 problems (~40 min)
print("Alpha sweep (100 problems each for speed):\n")
sweep = {}
sweep[0.0] = evaluate_steered(None, 0, "base, no steering (alpha=0)", n_eval=100)
for alpha in [1.0, 2.0, 4.0, 8.0]:
    sweep[alpha] = evaluate_steered(steer_vec, alpha, f"base + raw vector, alpha={alpha}", n_eval=100)

best_alpha = max(sweep, key=sweep.get)
print(f"\nBest alpha: {best_alpha} ({sweep[best_alpha]:.1%})")
print("If accuracy collapses at high alpha, that's the vector overwhelming the model — expected.")

Alpha sweep (100 problems each for speed):



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base, no steering (alpha=0)              6/100 = 6.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=1.0             4/100 = 4.0%


...

base + raw vector, alpha=2.0             7/100 = 7.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=4.0             14/100 = 14.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=8.0             17/100 = 17.0%

Best alpha: 8.0 (17.0%)
If accuracy collapses at high alpha, that's the vector overwhelming the model — expected.


In [9]:
for alpha in [12.0, 16.0, 24.0, 32.0]:
    sweep[alpha] = evaluate_steered(steer_vec, alpha, f"base + raw vector, alpha={alpha}", n_eval=100)
best_alpha = max(sweep, key=sweep.get)
print(f"\nBest alpha overall: {best_alpha} ({sweep[best_alpha]:.1%})")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=12.0            16/100 = 16.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=16.0            7/100 = 7.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=24.0            1/100 = 1.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector, alpha=32.0            1/100 = 1.0%

Best alpha overall: 8.0 (17.0%)


In [10]:
# Cell 7: EXPERIMENT B — SAE-filtered sparse steering (the plausibly-novel part)
# Train the round-4 SAE, find top-shifted features, rebuild the steering vector
# from ONLY those features' decoder directions.
import torch.nn as nn
import torch.nn.functional as F

D_IN = 896; D_SAE = D_IN * 8

class SAE(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.enc = nn.Linear(d_in, d_sae)
        self.dec = nn.Linear(d_sae, d_in)
    def forward(self, x):
        f = torch.relu(self.enc(x))
        return self.dec(f), f

all_acts = torch.cat([acts_base, acts_ft], 0)
scale = all_acts.norm(dim=1).mean()
train_data = (all_acts / scale).cuda()

torch.manual_seed(SEED)
sae = SAE(D_IN, D_SAE).cuda()
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)
print("Training SAE (~5 min)...")
n = train_data.shape[0]
for epoch in range(15):
    perm = torch.randperm(n, device="cuda")
    for i in range(0, n, 4096):
        x = train_data[perm[i:i+4096]]
        recon, feats = sae(x)
        loss = F.mse_loss(recon, x) + 5e-4 * feats.abs().mean()
        opt.zero_grad(); loss.backward(); opt.step()
print("SAE trained.")

with torch.no_grad():
    _, f_base = sae((acts_base / scale).cuda()); f_base = f_base.mean(0).cpu()
    _, f_ft   = sae((acts_ft / scale).cuda());   f_ft = f_ft.mean(0).cpu()
shift = f_ft - f_base

def sparse_vector(top_k):
    """Rebuild the steering vector from only the top_k most-shifted SAE features."""
    idx = shift.abs().topk(top_k).indices
    with torch.no_grad():
        W_dec = sae.dec.weight.cpu()              # [896, D_SAE]
        v = (W_dec[:, idx] @ shift[idx]) * scale  # back to activation units
    return v

for K in [10, 30, 100]:
    sv = sparse_vector(K)
    cos_to_raw = F.cosine_similarity(sv.unsqueeze(0), steer_vec.unsqueeze(0)).item()
    print(f"K={K:>4}: sparse vector norm {sv.norm():.3f}, cosine to raw vector {cos_to_raw:.3f}")

Training SAE (~5 min)...
SAE trained.
K=  10: sparse vector norm 0.154, cosine to raw vector 0.382
K=  30: sparse vector norm 0.233, cosine to raw vector 0.535
K= 100: sparse vector norm 0.358, cosine to raw vector 0.751


In [11]:
# Cell 8: Evaluate sparse steering at the best alpha from the sweep (~30 min)
print(f"Sparse steering at alpha={best_alpha}, 100 problems each:\n")
sparse_results = {}
for K in [10, 30, 100]:
    sv = sparse_vector(K)
    sparse_results[K] = evaluate_steered(sv, best_alpha, f"base + top-{K} SAE features", n_eval=100)

Sparse steering at alpha=8.0, 100 problems each:



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + top-10 SAE features               7/100 = 7.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + top-30 SAE features               4/100 = 4.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + top-100 SAE features              16/100 = 16.0%


In [12]:
# Cell 9: Final confirmation on the FULL 200 problems + scoreboard
print("Full 200-problem confirmation runs:\n")
final_base   = evaluate_steered(None, 0, "base (no steering)", n_eval=200)
final_raw    = evaluate_steered(steer_vec, best_alpha, f"base + raw vector (alpha={best_alpha})", n_eval=200)
best_K = max(sparse_results, key=sparse_results.get)
final_sparse = evaluate_steered(sparse_vector(best_K), best_alpha,
                                f"base + top-{best_K} SAE features (alpha={best_alpha})", n_eval=200)
final_ft     = evaluate_steered(None, 0, "full fine-tune (reference)", n_eval=200, use_adapter=True)

print("\n" + "="*62)
print("ROUND 5 SCOREBOARD — GSM8K, 200 held-out problems")
print("="*62)
print(f"{'base model, untouched':<45}{final_base:>8.1%}")
print(f"{'base + ONE raw steering vector':<45}{final_raw:>8.1%}")
print(f"{'base + top-' + str(best_K) + ' SAE features only':<45}{final_sparse:>8.1%}")
print(f"{'full fine-tune (1 GPU-hour of training)':<45}{final_ft:>8.1%}")
print("="*62)
gain_ft = final_ft - final_base
if gain_ft > 0:
    print(f"\nFraction of fine-tuning's gain recovered by:")
    print(f"  raw vector:    {max(0,(final_raw-final_base))/gain_ft:.0%}")
    print(f"  sparse vector: {max(0,(final_sparse-final_base))/gain_ft:.0%}")
print("""
Reading the result:
- Recovery >30%: a big chunk of fine-tuning IS one direction. Striking.
- Recovery ~5-20%: partial — steering captures the 'format/mode' part but
  not the computation. Still interesting: which part lives in the vector?
- Recovery ~0%: fine-tuning's effect is distributed/token-dependent;
  a constant vector can't express it. A clean negative result.
- Sparse ≈ raw with only ~30 features: the change is low-dimensional AND
  interpretable — that's the headline outcome if it happens.
""")

Full 200-problem confirmation runs:



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base (no steering)                       13/200 = 6.5%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + raw vector (alpha=8.0)            43/200 = 21.5%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base + top-100 SAE features (alpha=8.0)  34/200 = 17.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

full fine-tune (reference)               68/200 = 34.0%

ROUND 5 SCOREBOARD — GSM8K, 200 held-out problems
base model, untouched                            6.5%
base + ONE raw steering vector                  21.5%
base + top-100 SAE features only                17.0%
full fine-tune (1 GPU-hour of training)         34.0%

Fraction of fine-tuning's gain recovered by:
  raw vector:    55%
  sparse vector: 38%

Reading the result:
- Recovery >30%: a big chunk of fine-tuning IS one direction. Striking.
- Recovery ~5-20%: partial — steering captures the 'format/mode' part but
  not the computation. Still interesting: which part lives in the vector?
- Recovery ~0%: fine-tuning's effect is distributed/token-dependent;
  a constant vector can't express it. A clean negative result.
- Sparse ≈ raw with only ~30 features: the change is low-dimensional AND
  interpretable — that's the headline outcome if it happens.

